In [1]:
!pip install segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 73.2 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.

In [1]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

TRAIN_IMG_DIR = '/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Train/Train/Urban/images_png'
TRAIN_MASK_DIR = '/kaggle/input/datasets/mohammedjaveed/loveda-dataset/Train/Train/Urban/masks_png'

class UrbanDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        self.images = sorted([f for f in os.listdir(images_dir) if not f.startswith('.')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.images[idx])
        
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask.long()

train_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_dataset = UrbanDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

print("Конвейер данных успешно инициализирован.")
print(f"Общее количество загруженных изображений для обучения: {len(train_dataset)}")

Конвейер данных успешно инициализирован.
Общее количество загруженных изображений для обучения: 1156


In [6]:
import time
import torch
import torch.optim as optim
import torch.nn as nn
import segmentation_models_pytorch as smp

# Инициализация устройства и архитектуры FPN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Вычислительное устройство: {device}")

model_fpn = smp.FPN(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
).to(device)

optimizer = optim.Adam(model_fpn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
EPOCHS = 50

print("Начало процесса обучения модели FPN...")

# Цикл обучения
for epoch in range(EPOCHS):
    model_fpn.train()
    total_loss = 0
    
    # Предполагается, что объект train_loader инициализирован ранее
    for images, masks in train_loader: 
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model_fpn(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Эпоха [{epoch+1}/{EPOCHS}] | Среднее значение функции потерь (Loss): {avg_loss:.4f}")

# Сохранение весов обученной модели
torch.save(model_fpn.state_dict(), 'best_model_fpn.pth')
print("Обучение завершено. Веса модели успешно сохранены в файл best_model_fpn.pth.")

Вычислительное устройство: cuda


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Начало процесса обучения модели FPN...
Эпоха [1/50] | Среднее значение функции потерь (Loss): 1.7606
Эпоха [2/50] | Среднее значение функции потерь (Loss): 1.2018
Эпоха [3/50] | Среднее значение функции потерь (Loss): 1.0414
Эпоха [4/50] | Среднее значение функции потерь (Loss): 0.9328
Эпоха [5/50] | Среднее значение функции потерь (Loss): 0.8627
Эпоха [6/50] | Среднее значение функции потерь (Loss): 0.8132
Эпоха [7/50] | Среднее значение функции потерь (Loss): 0.7547
Эпоха [8/50] | Среднее значение функции потерь (Loss): 0.8184
Эпоха [9/50] | Среднее значение функции потерь (Loss): 0.7400
Эпоха [10/50] | Среднее значение функции потерь (Loss): 0.6975
Эпоха [11/50] | Среднее значение функции потерь (Loss): 0.6484
Эпоха [12/50] | Среднее значение функции потерь (Loss): 0.6058
Эпоха [13/50] | Среднее значение функции потерь (Loss): 0.5833
Эпоха [14/50] | Среднее значение функции потерь (Loss): 0.5569
Эпоха [15/50] | Среднее значение функции потерь (Loss): 0.5879
Эпоха [16/50] | Среднее з